# PayShield AI — Threshold Optimization

## AI-Powered Payment Success Optimization

### Objective

Determine the probability threshold at which PayShield
should classify a monitoring window as high payment-risk.

The default machine-learning threshold is 0.50, but this
may not be appropriate for an alerting system.

We will compare different thresholds using:

- Precision
- Recall
- F1-score

The goal is to reduce false alerts while still detecting
important payment-system problems.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

In [2]:
df = pd.read_csv(
    "../data/processed/ml_dataset.csv"
)

print("Dataset shape:", df.shape)

Dataset shape: (50944, 22)


In [3]:
X = df.drop(columns=["risk_target"])
y = df["risk_target"]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (50944, 21)
Target: (50944,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (40755, 21)
Testing: (10189, 21)


In [5]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [6]:
risk_probability = rf_model.predict_proba(
    X_test
)[:, 1]

print("First 20 risk probabilities:")
print(risk_probability[:20])

First 20 risk probabilities:
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [7]:
thresholds = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]

threshold_results = []

for threshold in thresholds:

    predictions = (
        risk_probability >= threshold
    ).astype(int)

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_df = pd.DataFrame(
    threshold_results
)

threshold_df

,threshold,precision,recall,f1
0,0.1,0.770492,0.839286,0.803419
1,0.2,0.843137,0.767857,0.803738
2,0.3,0.893617,0.750000,0.815534
3,0.4,0.931818,0.732143,0.820000
4,0.5,0.931818,0.732143,0.820000
5,0.6,0.976190,0.732143,0.836735
6,0.7,0.975000,0.696429,0.812500
7,0.8,0.975000,0.696429,0.812500
8,0.9,0.969697,0.571429,0.719101


In [8]:
threshold_df.round(3)

,threshold,precision,recall,f1
0,0.1,0.770,0.839,0.803
1,0.2,0.843,0.768,0.804
2,0.3,0.894,0.750,0.816
3,0.4,0.932,0.732,0.820
4,0.5,0.932,0.732,0.820
5,0.6,0.976,0.732,0.837
6,0.7,0.975,0.696,0.812
7,0.8,0.975,0.696,0.812
8,0.9,0.970,0.571,0.719


In [9]:
best_f1_row = threshold_df.loc[
    threshold_df["f1"].idxmax()
]

print("Best threshold based on F1:")
print(best_f1_row)

Best threshold based on F1:
threshold    0.600000
precision    0.976190
recall       0.732143
f1           0.836735
Name: 5, dtype: float64


In [10]:
threshold_results = []

for threshold in thresholds:

    predictions = (
        risk_probability >= threshold
    ).astype(int)

    precision = precision_score(
        y_test,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        zero_division=0
    )

    alert_count = predictions.sum()

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "alerts": alert_count
    })

threshold_df = pd.DataFrame(
    threshold_results
)

threshold_df.round(3)

,threshold,precision,recall,f1,alerts
0,0.1,0.770,0.839,0.803,61
1,0.2,0.843,0.768,0.804,51
2,0.3,0.894,0.750,0.816,47
3,0.4,0.932,0.732,0.820,44
4,0.5,0.932,0.732,0.820,44
5,0.6,0.976,0.732,0.837,42
6,0.7,0.975,0.696,0.812,40
7,0.8,0.975,0.696,0.812,40
8,0.9,0.970,0.571,0.719,33


In [11]:
precision_target = 0.80

eligible = threshold_df[
    threshold_df["precision"] >= precision_target
]

if len(eligible) > 0:

    selected_threshold = eligible.iloc[0]["threshold"]

else:

    selected_threshold = (
        threshold_df.loc[
            threshold_df["f1"].idxmax(),
            "threshold"
        ]
    )

print(
    "Selected PayShield threshold:",
    selected_threshold
)

Selected PayShield threshold: 0.2


In [12]:
final_predictions = (
    risk_probability >= selected_threshold
).astype(int)

final_precision = precision_score(
    y_test,
    final_predictions,
    zero_division=0
)

final_recall = recall_score(
    y_test,
    final_predictions,
    zero_division=0
)

final_f1 = f1_score(
    y_test,
    final_predictions,
    zero_division=0
)

print("PayShield Threshold:", selected_threshold)
print("Precision:", final_precision)
print("Recall:", final_recall)
print("F1:", final_f1)

PayShield Threshold: 0.2
Precision: 0.8431372549019608
Recall: 0.7678571428571429
F1: 0.8037383177570093


In [13]:
def get_risk_level(probability):

    if probability >= 0.80:
        return "HIGH"

    elif probability >= 0.50:
        return "MEDIUM"

    else:
        return "LOW"

In [14]:
prediction_results = X_test.copy()

prediction_results["actual_risk"] = y_test.values

prediction_results["risk_probability"] = (
    risk_probability
)

prediction_results["risk_level"] = (
    prediction_results["risk_probability"]
    .apply(get_risk_level)
)

prediction_results.head(20)

,transaction_count,failure_rate,timeout_rate,avg_latency,max_latency,p95_latency,bank_error_rate,avg_amount,max_amount,hour,...,previous_timeout_rate,failure_rate_change,latency_change,timeout_rate_change,rolling_failure_rate,rolling_latency,rolling_timeout_rate,actual_risk,risk_probability,risk_level
35855,4,0.0,0.0,1064.240000,1861.51,1748.2105,0.0,204.727500,309.03,20,...,0.0,0.0,-844.200000,0.0,0.000000,1326.718333,0.0,0,0.0,LOW
24059,2,0.0,0.0,936.830000,1294.56,1258.7870,0.0,427.075000,710.07,17,...,0.0,0.0,-174.122500,0.0,0.000000,1301.178611,0.0,0,0.0,LOW
44616,2,0.0,0.0,965.595000,1156.23,1137.1665,0.0,981.765000,1404.47,15,...,0.0,0.0,-357.960000,0.0,0.000000,1148.745000,0.0,0,0.0,LOW
15967,2,0.0,0.0,1571.595000,2815.04,2690.6955,0.0,211.040000,340.84,8,...,0.0,0.0,136.498333,0.0,0.000000,1317.770556,0.0,0,0.0,LOW
20058,1,0.0,0.0,903.040000,903.04,903.0400,0.0,2002.680000,2002.68,2,...,0.0,0.0,490.770000,0.0,0.000000,744.826667,0.0,0,0.0,LOW
28352,2,0.0,0.0,954.760000,1155.62,1135.5340,0.0,738.465000,1299.34,20,...,0.0,0.0,-208.240000,0.0,0.000000,1013.668889,0.0,0,0.0,LOW
25088,2,0.0,0.0,1554.915000,1665.24,1654.2075,0.0,556.965000,805.48,18,...,0.0,0.0,553.901667,0.0,0.000000,1201.502778,0.0,0,0.0,LOW
43096,5,0.0,0.0,967.940000,1330.08,1287.1660,0.0,240.650000,311.48,15,...,0.0,0.0,62.000000,0.0,16.666667,975.850000,0.0,0,0.0,LOW
7043,4,0.0,0.0,1276.027500,1725.03,1717.5240,0.0,157.055000,297.82,20,...,0.0,0.0,334.597500,0.0,0.000000,1083.225833,0.0,0,0.0,LOW
44858,1,0.0,0.0,1165.040000,1165.04,1165.0400,0.0,59.020000,59.02,22,...,0.0,0.0,-102.930000,0.0,0.000000,1341.693333,0.0,0,0.0,LOW


In [15]:
print(
    prediction_results["risk_level"]
    .value_counts()
)

risk_level
LOW       10145
HIGH         40
MEDIUM        4
Name: count, dtype: int64


In [16]:
prediction_results.to_csv(
    "../data/processed/payShield_predictions.csv",
    index=False
)

print(
    "PayShield predictions saved successfully!"
)

PayShield predictions saved successfully!
